# Homework 3: 文字分類，使用 PyTorch 與 HuggingFace Transformers

## HuggingFace Transformers 介紹
- HuggingFace Transformers 是一個現代自然語言處理的函式庫，實作了許多基於 Transformer 架構的預訓練模型，如 BERT、GPT-2、Llama 等。
- 它提供了簡單易用的 API，讓使用者能夠輕鬆地載入預訓練模型，並進行微調 (fine-tuning) 以適應特定任務，如文字分類、命名實體識別等。
- 官方網站與文件：https://huggingface.co/docs/transformers/index
- 由於 Deep Learning 理論課程緊湊，Transformers 實作教學部分請看影片：
    - 長庚版本 (47+51分鐘)：
        - https://youtu.be/tUO-XqMmWPU?si=oE6N5AJgLdfd2s6b
        - https://youtu.be/KxIwjub6tZc?si=rCWm07fNBoAl12VX
    - 清大版本 (約90分鐘)：
        - https://www.youtube.com/watch?v=VErSpYgZGiw

In [1]:
!pip install datasets # HuggingFace Datasets

In [2]:
import torch

# We need to first test if the Colab environment is already in the use of GPU.
# Otherwise, we might restart the Colab environment later.
torch.cuda.is_available() # The printed result must be `True`.

True

In [3]:
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    EvalPrediction,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

In [4]:
DATA_NAME = "dair-ai/emotion"
MODEL_NAME = "bert-base-uncased" # You can try other models.

In [ ]:
train_data = load_dataset(DATA_NAME, split="train")
valid_data = load_dataset(DATA_NAME, split="validation")
test_data = load_dataset(DATA_NAME, split="test")

In [ ]:
# Check the classes in the dataset
print(np.unique(train_data["label"]))

num_labels = len(np.unique(train_data["label"]))

In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# 檢查特殊 token ID (special token ids)
for value in tokenizer.special_tokens_map.values():
    print(f"token {value} 的 ID 為：{tokenizer.convert_tokens_to_ids(value)}")

In [ ]:
# TODO1: Pre-process sentences with `tokenizer`.
# You should set up the parameter for `tokenizer`
# to cut off the excessive sentences that are longer and the max length of BERT.

# Please note that we don't need to perform padding at this step,
# because we will perform dynamic padding later with `collate_fn`.

def preprocess_function(examples):
    return tokenizer(# Write your code here)

In [ ]:
encoded_train = train_data.map(preprocess_function, batched=True, remove_columns=["text"])
encoded_valid = valid_data.map(preprocess_function, batched=True, remove_columns=["text"])
encoded_test = test_data.map(preprocess_function, batched=True, remove_columns=["text"])

# Please note that`batched` != batch_size
# `batched=True` means to process data in batches with the map function
# Usually, `batched=True` computes faster.

In [ ]:
# Observation with a tokenized example

first_token_ids = encoded_test['input_ids'][0]
print(f"第一筆資料被轉換成 IDs 的結果: {first_token_ids}")
print(f"把 IDs 換回原本文字: {tokenizer.decode(first_token_ids, skip_special_tokens=False)}")
print(f"原始文字: {test_data['text'][0]}")

# You can also set `padding=True` in `preprocess_function` to observe the difference.

In [ ]:
# TODO2: Finish `collate_fn` for dynamic padding.
# `DataLoader` gives `collate_fn` a list of samples (dicts), e.g.
#   [{"label": 0, "input_ids": [101, 10047, ...], "token_type_ids": [...], "attention_mask": [...]}, ...]
# `collate_fn` should return a dict of tensors that can be fed into the model with `model(**batch)`:
#   "input_ids":      LongTensor (B, L), padded with `tokenizer.pad_token_id`, L = the longest length in this batch
#   "attention_mask": LongTensor (B, L), 1 for real tokens, 0 for padding
#   "labels":         LongTensor (B,)   (note: the model expects the key `labels`, not `label`)
#
# 提示 (依序完成)：
#   1. 把每一筆的 `input_ids` 轉成 tensor 放進一個 list；把所有 `label` 轉成一個 tensor
#   2. 用 `pad_sequence(..., batch_first=True, padding_value=tokenizer.pad_token_id)` 補到這個 batch 中最長的長度
#   3. `attention_mask`：`input_ids` 中不等於 `tokenizer.pad_token_id` 的位置為 1，其餘為 0
#      (提示：比較運算 `!=` 會得到 bool tensor，再用 `.long()` 轉成整數)
# 我們不需要 `token_type_ids`：只有一個句子時，BERT 會自動把它設為全 0。

def collate_fn(batch):
    # Write your code here

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
# Observation: dynamic padding with `collate_fn`
# The lengths are differnt in different batches.

tmp_batch1 = collate_fn([encoded_test[i] for i in range(0, 5)])
print(len(tmp_batch1["input_ids"][0]))

tmp_batch2 = collate_fn([encoded_test[i] for i in range(5, 10)])
print(len(tmp_batch2["input_ids"][0]))

In [ ]:
# Set up DataLoaders

train_loader = DataLoader(encoded_train, batch_size=16, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(encoded_valid, batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(encoded_test, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [ ]:
# Set up the BERT model for sequence classification

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
)

In [ ]:
# Move the model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# Set up the optimizer and the learning rate scheduler (the same defaults as `Trainer`)

NUM_EPOCHS = 3
LEARNING_RATE = 5e-5
WARMUP_STEPS = 500

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=NUM_EPOCHS * len(train_loader),
)

In [ ]:
def compute_metrics(model_eval_pred):
    preds = model_eval_pred.predictions.argmax(axis=1)
    labels = model_eval_pred.label_ids

    # TODO3: Write scoring functions to get acc, precision, recall, and f1-score.
    # Write your code here

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
# Evaluation: collect the predictions on a dataset, then call `compute_metrics`

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels, total_loss = [], [], 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item() * len(batch["labels"])
        all_logits.append(outputs.logits.cpu())
        all_labels.append(batch["labels"].cpu())
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    metrics = compute_metrics(EvalPrediction(predictions=logits, label_ids=labels))
    return {"loss": total_loss / len(labels), **metrics}

In [ ]:
# TODO4: Finish the PyTorch training loop.
# 每一個 batch 請依序完成：
#   1. 清空上一步的 gradient：`optimizer.zero_grad()`
#   2. Forward：`outputs = model(**batch)`
#      (因為 batch 裡有 "labels"，`outputs.loss` 就是 cross-entropy loss)
#   3. Backward：`loss.backward()`
#   4. Gradient clipping (與 `Trainer` 預設相同)：`torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)`
#   5. 更新參數與 learning rate：`optimizer.step()`、`scheduler.step()`

history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        # Write your code here
        running_loss += loss.item()

    valid_metrics = evaluate(model, valid_loader)
    history.append({"epoch": epoch, "train_loss": running_loss / len(train_loader),
                    **{f"valid_{k}": v for k, v in valid_metrics.items()}})
    print(f"Epoch {epoch} | train loss {history[-1]['train_loss']:.4f} | "
          f"valid loss {valid_metrics['loss']:.4f} | valid acc {valid_metrics['accuracy']:.4f} | "
          f"valid f1 {valid_metrics['f1']:.4f}")

In [ ]:
metrics = evaluate(model, test_loader)

In [ ]:
print(metrics)

In [ ]:
df = pd.DataFrame(metrics.items(), columns=["Metric", "Value"])
print(df.to_markdown(index=False))